# Panel de balance mensual — desde 2020

Construye el panel largo `data/interim/paneles_largos/panel_balance_mensual.parquet` a partir de `bal_hist/balhist.txt` del dump IEF más reciente, filtrando al período en moneda homogénea (desde enero 2020).

Estructura: una fila por `(codigo_entidad, yyyymm, codigo_cuenta)` con el saldo en pesos nominales tal como lo reporta el BCRA.

Es el **panel principal** del proyecto. Todas las regresiones del evento del blanqueo (ago-2024 ± 12 meses) parten de acá.

Decisiones metodológicas aplicadas (ver `docs/notas/metodologia_paneles.md`):
- §3.1: se trabaja "bank-as-is" — los códigos `AAxxx` se separan a `panel_balance_agregados.parquet` en otro notebook.
- §3.3: el corte en 2020-01 responde al cambio a moneda homogénea (Com. "A" 6651).
- §3.4: no se hace zero-fill; las combinaciones ausentes se interpretan como cero en los agregados posteriores.
- §3.7: los saldos quedan en pesos nominales (la conversión a USD ocurre en notebooks de análisis).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import RAW, INTERIM, DIMENSIONES, PANELES, REPO

import pandas as pd
import duckdb

BALHIST = RAW / "bcra_ief/202601/Entfin/Tec_Cont/bal_hist/balhist.txt"
OUT = PANELES / "panel_balance_mensual.parquet"
OUT.parent.mkdir(parents=True, exist_ok=True)

FECHA_CORTE = 202001  # yyyymm mínimo

## Lectura

Archivo TSV con cuatro columnas, sin header, codificación ISO-8859-1. Los campos alfanuméricos vienen entre comillas; el saldo es numérico en miles de pesos.

In [ ]:
raw = pd.read_csv(
    BALHIST,
    sep="\t",
    header=None,
    names=["codigo_entidad", "yyyymm", "codigo_cuenta", "saldo_miles_pesos"],
    encoding="ISO-8859-1",
    dtype={"codigo_entidad": str, "yyyymm": str, "codigo_cuenta": str, "saldo_miles_pesos": float},
)
print(f"Filas leídas: {len(raw):,}")
print(f"Rango temporal: {raw['yyyymm'].min()} - {raw['yyyymm'].max()}")
raw.head()

## Filtros

Dos filtros:
1. Período: `yyyymm >= 202001` para quedarse en moneda homogénea.
2. Entidades: se excluyen los agrupamientos (códigos `AAxxx`); van a un panel separado.

In [ ]:
raw["yyyymm_int"] = raw["yyyymm"].astype(int)
panel = raw[raw["yyyymm_int"] >= FECHA_CORTE].copy()
print(f"Post-filtro temporal (>= {FECHA_CORTE}): {len(panel):,} filas")

panel = panel[~panel["codigo_entidad"].str.startswith("AA")].copy()
print(f"Post-filtro entidades (sin agrupamientos): {len(panel):,} filas")

## Columnas derivadas

Agrega `fecha` como último día del mes, para facilitar joins temporales con series diarias o trimestrales.

In [ ]:
panel["fecha"] = pd.to_datetime(panel["yyyymm"], format="%Y%m") + pd.offsets.MonthEnd(0)
panel["saldo"] = panel["saldo_miles_pesos"] * 1000  # convierto a pesos (no miles)

columnas = ["codigo_entidad", "yyyymm", "fecha", "codigo_cuenta", "saldo"]
panel = panel[columnas].copy()
panel["yyyymm"] = panel["yyyymm"].astype(int)
panel.head()

## Persistencia

In [ ]:
panel.to_parquet(OUT, index=False)
print(f"Escrito: {OUT.relative_to(REPO)}")
print(f"Filas: {len(panel):,}")

## Validaciones

Chequeos de integridad: rango de fechas, unicidad de la clave, número de entidades y cuentas.

In [ ]:
assert panel["yyyymm"].min() >= FECHA_CORTE, "Quedaron filas antes del corte"
assert panel["yyyymm"].max() >= 202501, "Se esperaba cobertura hasta 2025 al menos"
assert not panel["codigo_entidad"].str.startswith("AA").any(), "Quedaron agrupamientos"
assert panel[["codigo_entidad", "yyyymm", "codigo_cuenta"]].duplicated().sum() == 0, "Clave duplicada"

resumen = {
    "filas": len(panel),
    "entidades": panel["codigo_entidad"].nunique(),
    "cuentas": panel["codigo_cuenta"].nunique(),
    "meses": panel["yyyymm"].nunique(),
    "fecha_min": panel["yyyymm"].min(),
    "fecha_max": panel["yyyymm"].max(),
}
print("Validaciones OK")
for k, v in resumen.items():
    print(f"  {k}: {v:,}")

## Chequeo rápido con DuckDB

Suma del saldo por mes para verificar que la serie tiene la forma esperada.

In [ ]:
duckdb.sql(f"""
    select yyyymm, count(*) as filas, sum(saldo) / 1e12 as saldo_total_billones
    from '{OUT}'
    group by yyyymm
    order by yyyymm
    limit 10
""").df()

## Muestra: cuentas CERA (Ley 27.743)

Ejemplo de cruce con `dim_cuentas` para aislar las cuatro cuentas del blanqueo.

In [ ]:
DIM_CUENTAS = DIMENSIONES / "dim_cuentas.parquet"

duckdb.sql(f"""
    select p.codigo_entidad, p.yyyymm, c.denominacion, p.saldo / 1e9 as saldo_miles_millones
    from '{OUT}' p
    join '{DIM_CUENTAS}' c using (codigo_cuenta)
    where c.es_cera
      and p.saldo != 0
    order by p.yyyymm desc, p.codigo_entidad
    limit 15
""").df()